## DECISION TREE

In [ ]:
import pandas as aju
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier,plot_tree
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.metrics import confusion_matrix
import seaborn as sn
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = aju.read_csv("../DATASETS/ML_08_DecisionTree_Salaries.csv")
df.head()

In [ ]:
le = LabelEncoder()
df['company'] = le.fit_transform(df['company'])

In [ ]:
df['job'] = le.fit_transform(df['job'])
df

In [ ]:
df['degree'] = le.fit_transform(df['degree'])
df

In [ ]:
x = df.drop('salary_more_then_100k',axis = 'columns')
y = df['salary_more_then_100k']

In [ ]:
x.columns

In [ ]:
model = DecisionTreeClassifier()
dcc = model.fit(x,y)

In [ ]:
plt.figure(figsize = (15,8))
plot_tree(dcc,filled = True,class_names = ["Above 100k","Below 100k"],rounded = True,feature_names = x.columns )
plt.show()

In [ ]:
model.score(x,y)

In [ ]:
model.predict([[2,2,1]])

In [ ]:
model.predict([[2,1,0]])

In [ ]:
model.predict([[1,1,0]])

## DECISION TREE PRUNING

In [ ]:
file = aju.read_csv('../DATASETS/ML_08_DecisionTree_Titanic.csv')
file.head()

In [ ]:
file['Survived'].value_counts()

In [ ]:
file.info()

In [ ]:
file.describe()

In [ ]:
file_df = file[['Survived','Pclass','Age','Fare','Sex']]
file_df

In [ ]:
file_df['Age'].isnull().sum()

In [ ]:
mean_age = file_df['Age'].mean()

In [ ]:
file_df['Age'] = file_df['Age'].fillna(mean_age)

In [ ]:
file_df

In [ ]:
file_df['Age'].isnull().sum()

In [ ]:
file_df.Sex = file_df.Sex.map({'male':1,'female':2})
file_df

In [ ]:
x1 = file_df.drop('Survived',axis = 'columns')
y1 = file_df.Survived

In [ ]:
x_train1,x_test1,y_train1,y_test1 = train_test_split(x1,y1,test_size = 0.2,random_state = 33)

In [ ]:
model1 = DecisionTreeClassifier(max_depth = 4)

In [ ]:
dcc_titanic = model1.fit(x_train1,y_train1)

In [ ]:
model1.score(x_test1,y_test1)

In [ ]:
y_pred1 = model1.predict(x_test1)

In [ ]:
cm = confusion_matrix(y_test1,y_pred1)
cm

In [ ]:
sn.heatmap(cm,annot = True)

In [ ]:
plt.figure(figsize = (15,8))
plot_tree(dcc_titanic,filled = True,class_names = ["Not survived","Survived"],rounded = True,feature_names = x1.columns )
plt.show()

In [ ]:
path = dcc_titanic.cost_complexity_pruning_path(x_train1,y_train1)

In [ ]:
path.ccp_alphas

In [ ]:
ccp_alpha = path.ccp_alphas

In [ ]:
ccp_alpha = ccp_alpha[:-1]

In [ ]:
ccp_alpha

In [ ]:
clf_dts = []
for i in ccp_alpha:
    clf_dt = DecisionTreeClassifier(random_state = 99,ccp_alpha = i)
    clf_dt.fit(x_train1,y_train1)
    clf_dts.append(clf_dt)

In [ ]:
clf_dts

In [ ]:
train_score = [i.score(x_train1,y_train1) for i in clf_dts]

In [ ]:
train_score

In [ ]:
test_score = [i.score(x_test1,y_test1) for i in clf_dts]

In [ ]:
test_score

In [ ]:
alpha_loop = []
for i in ccp_alpha:
    clf_dt = DecisionTreeClassifier(random_state = 99,ccp_alpha = i)
    score = cross_val_score(clf_dt,x_train1,y_train1,cv = 5)
    alpha_loop.append([i,np.mean(score),np.std(score)])

In [ ]:
alpha_loop

In [ ]:
res = aju.DataFrame(alpha_loop,columns = ['alpha','mean_score','std'])
res

In [ ]:
res[res.mean_score == res.mean_score.max()]

In [ ]:
ideal_ccp = 0.009774

In [ ]:
dcc = DecisionTreeClassifier(random_state = 99,ccp_alpha = ideal_ccp)

In [ ]:
dcc.fit(x_train1,y_train1)

In [ ]:
plot_tree(dcc,filled = True,class_names = ['s','n'],feature_names = x_train1.columns)
plt.show()